In [8]:
from dotenv import load_dotenv
load_dotenv()

True

In [9]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="gpt-6-luna",
)

### Static Templates

In [10]:
# Chat models accept raw lists of role-content dictionaries directly.
# Valid standard roles are "system", "user", and "assistant".
prompt = [
    {"role": "system", "content": "You are a senior technical writer."},
    {"role": "user", "content": "Explain the difference between concurrency and parallelism in one sentence."}
]

res = llm.invoke(prompt)
print(res.content)

Concurrency is managing multiple tasks in overlapping time periods, while parallelism is executing multiple tasks at the same time.


### Dynamic Templates

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

In [12]:
# Parameterized prompt with explicit variables
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a professional localization specialist. Translate the text into {language}."),
        ("user", "{query}")
    ]
)

print(prompt)

input_variables=['language', 'query'] input_types={} partial_variables={} messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['language'], input_types={}, partial_variables={}, template='You are a professional localization specialist. Translate the text into {language}.'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['query'], input_types={}, partial_variables={}, template='{query}'), additional_kwargs={})]


In [13]:
# Method A: format_messages returns a list of BaseMessage objects
final_prompt = prompt.format_messages(language="German", query="The database connection timed out.")
print(final_prompt)
res = llm.invoke(final_prompt)
print(res.content)

[SystemMessage(content='You are a professional localization specialist. Translate the text into German.', additional_kwargs={}, response_metadata={}), HumanMessage(content='The database connection timed out.', additional_kwargs={}, response_metadata={})]
Zeitüberschreitung bei der Datenbankverbindung.


In [14]:
# Method B: prompt.invoke produces a ChatPromptValue wrapper
prompt_value = prompt.invoke({"language": "German", "query": "The database connection timed out."})
print(prompt_value)
res = llm.invoke(prompt.invoke({"language": "German", "query": "The database connection timed out."}))
print(res.content)

messages=[SystemMessage(content='You are a professional localization specialist. Translate the text into German.', additional_kwargs={}, response_metadata={}), HumanMessage(content='The database connection timed out.', additional_kwargs={}, response_metadata={})]
Zeitüberschreitung bei der Verbindung zur Datenbank.


### Chaining with LangChain Expression Language (LCEL)

In [17]:
# Basic sequence: Prompt -> LLM
chain = prompt | llm
res = chain.invoke({"language": "Japanese", "query": "The deployment was successful."})
print(res)

content='デプロイは成功しました。' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 48, 'prompt_tokens': 28, 'total_tokens': 76, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 32, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': 0, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-6-luna', 'system_fingerprint': None, 'id': 'chatcmpl-ESHZK9IuzOflUzoeMdOwvpUBBjrtR', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--01a0dcb7-cdcc-78c0-ab91-5a1670d41403-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 28, 'output_tokens': 48, 'total_tokens': 76, 'input_token_details': {'audio': 0, 'cache_read': 0, 'cache_creation': 0}, 'output_token_details': {'audio': 0, 'reasoning': 32}}


In [18]:
# Sequence with output parsing: Prompt -> LLM -> Parser
from langchain_core.output_parsers import StrOutputParser
out = StrOutputParser()
chain = prompt | llm | out
res = chain.invoke({"language": "Japanese", "query": "The deployment was successful."})
print(res)

デプロイは成功しました。


In [20]:
# Custom transformation step
def to_upper(result: str) -> str:
    return result.upper()

chain = prompt | llm | out | to_upper
res = chain.invoke({"language": "English", "query": "デプロイは成功しました。"})
print(res)

THE DEPLOYMENT WAS SUCCESSFUL.
